# Unit 3 – Ex 3: Multiple linear regression
`cars.csv` isn't in the unit folder. This is the 36-car Volume/Weight/CO2 dataset the notebook was written for: it reproduces the notebook's coefficients (0.00755095, 0.00780526) and its 107.2 g prediction exactly.

(The file `Unit03 cars.csv` on the course server is a different dataset – auto-mpg, with no `Weight`/`Volume`/`CO2` columns – so the code won't run on it as-is.)

In [1]:
import os, pandas
from sklearn import linear_model

# use the local cars.csv if it is next to the notebook, otherwise the copy on the ePortfolio (works in Colab)
CSV = "cars.csv" if os.path.exists("cars.csv") else "https://raw.githubusercontent.com/aldanamaldosari/eportfolio/main/machine-learning/notebooks/cars.csv"
df = pandas.read_csv(CSV)

X = df[['Weight', 'Volume']]
y = df['CO2']

regr = linear_model.LinearRegression()
regr.fit(X, y)

# predict CO2 for weight 2300kg, volume 1300cm3
# (passing a DataFrame with column names avoids sklearn's feature-name warning)
predictedCO2 = regr.predict(pandas.DataFrame([[2300, 1300]], columns=['Weight', 'Volume']))
print(predictedCO2)

[107.2087328]


# Coefficient

In [2]:
print(regr.coef_)

[0.00755095 0.00780526]


In [3]:
predictedCO2 = regr.predict(pandas.DataFrame([[3300, 1300]], columns=['Weight', 'Volume']))
print(predictedCO2)

[114.75968007]


## Experiment 1 – How much does each variable explain?
Compare R² for Weight alone, Volume alone and both together, and the correlation between the two inputs.

In [4]:
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr
rows = []
for cols in (['Weight'], ['Volume'], ['Weight', 'Volume']):
    m = linear_model.LinearRegression().fit(df[cols], y)
    rows.append({'predictors': ' + '.join(cols), 'R²': m.score(df[cols], y),
                 'coefficients': np.round(m.coef_, 5), 'intercept': round(m.intercept_, 2)})
print('Correlation Weight vs Volume: r = %.3f' % pearsonr(df.Weight, df.Volume)[0])
print('Correlation Weight vs CO2:    r = %.3f' % pearsonr(df.Weight, y)[0])
print('Correlation Volume vs CO2:    r = %.3f' % pearsonr(df.Volume, y)[0])
pandas.DataFrame(rows)

Correlation Weight vs Volume: r = 0.754
Correlation Weight vs CO2:    r = 0.552
Correlation Volume vs CO2:    r = 0.592


,predictors,R²,coefficients,intercept
0,Weight,0.304870,[0.017],80.06
1,Volume,0.350561,[0.01135],83.75
2,Weight + Volume,0.376556,"[0.00755, 0.00781]",79.69


## Experiment 2 – Change the units / scale
If volume is recorded in litres instead of cm³, its coefficient gets 1000× bigger, but the model's predictions stay the same. So raw coefficients can't be used to say which variable 'matters more'. Standardising the inputs puts them on the same scale.

In [5]:
from sklearn.preprocessing import StandardScaler
df2 = df.assign(Volume_L=df.Volume/1000)
m_l = linear_model.LinearRegression().fit(df2[['Weight','Volume_L']], y)
print('Coefficients with Volume in litres:', m_l.coef_.round(5))
print('Prediction (2300kg, 1.3L):', m_l.predict(pandas.DataFrame([[2300,1.3]], columns=['Weight','Volume_L'])).round(3))

Z = StandardScaler().fit_transform(df[['Weight','Volume']])
m_z = linear_model.LinearRegression().fit(Z, y)
print('Standardised coefficients (CO2 change per 1 SD):  Weight=%.3f  Volume=%.3f' % tuple(m_z.coef_))

Coefficients with Volume in litres: [7.55000e-03 7.80526e+00]
Prediction (2300kg, 1.3L): [107.209]
Standardised coefficients (CO2 change per 1 SD):  Weight=1.803  Volume=2.994


## Experiment 3 – Change individual data points
Remove the heaviest/biggest cars, or add one unusual car, and refit.

In [6]:
def fit(d):
    m = linear_model.LinearRegression().fit(d[['Weight','Volume']], d['CO2'])
    p = m.predict(pandas.DataFrame([[2300,1300]], columns=['Weight','Volume']))[0]
    return {'n': len(d), 'coef Weight': m.coef_[0], 'coef Volume': m.coef_[1],
            'R²': m.score(d[['Weight','Volume']], d['CO2']), 'pred (2300kg,1300cm³)': p}
extra = pandas.DataFrame([{'Car':'Test','Model':'Odd','Volume':1000,'Weight':2000,'CO2':90}])
scen = {
    'original': df,
    'drop cars > 1600 kg': df[df.Weight <= 1600],
    'drop Mercedes SLK (2500cm³)': df[df.Model != 'SLK'],
    'add heavy, low-CO2 car': pandas.concat([df, extra], ignore_index=True),
}
pandas.DataFrame({k: fit(v) for k, v in scen.items()}).T.round(4)

,n,coef Weight,coef Volume,R²,"pred (2300kg,1300cm³)"
original,36.0,0.0076,0.0078,0.3766,107.2087
drop cars > 1600 kg,32.0,-0.0040,0.0095,0.2505,93.6857
drop Mercedes SLK (2500cm³),35.0,0.0127,0.0030,0.3398,113.4534
"add heavy, low-CO2 car",37.0,0.0012,0.0115,0.3842,99.4807


### What changed
* Weight and Volume are moderately correlated with each other, and each one alone explains only about a third of the variance in CO2 (R² 0.30 and 0.35). Together they explain about 38%, which adds little because the two inputs overlap. Most of the variation is still unexplained.
* Changing units moves the coefficients by factors of 1000 but leaves predictions identical. Standardised coefficients put them on a fair footing: one standard deviation of Volume is worth about 3.0 g CO2 against about 1.8 g for Weight.
* Note that 2300 kg is heavier than any car in the data (max 1746 kg), so the 107 g / 114.8 g predictions are **extrapolations**.
* Removing the four cars over 1600 kg makes the Weight coefficient *negative*. Removing just the Mercedes SLK, or adding one unusual car, moves the prediction by 6–14 g. With 36 rows and two overlapping inputs, the model is fragile.